# Experiment 12: deep-learning mixture of experts

Three embedding-based tabular neural networks are trained on the full dataset through chronological OOF folds. Numeric attributes—including `approval_date_num`—are median-imputed and standardized using fold-training rows only. A neural soft gate blends their OOF predictions using block with town fallback.

**Executed result:** the wide MLP, regularized MLP, and FT-Transformer scored **496.308**, **482.871**, and **476.979 RMSE**. Their equal blend improved to **471.168**, while the block→town neural gate scored **475.130** with mean weights **0.351 / 0.526 / 0.123**. Fixed averaging was the best deep-learning result and outperformed learned gating, but remained behind the boosted-tree ensemble.

In [1]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
configured_root = os.environ.get('KDDM_PROJECT_ROOT')
roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
project_root = next((p for p in roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
sys.path.insert(0, str(project_root / 'exploration' / 'metul'))
from preprocessing import RentalPreprocessor, load_datasets
TARGET = 'MONTHLY_RENT'
EPOCHS = int(os.environ.get('DL_MOE_EPOCHS', '80'))
BATCH = 1024
N_SPLITS = 3
torch.manual_seed(42)
np.random.seed(42)
CATS = ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL']
NUMS = ['FLOOR_AREA_SQM', 'approval_date_num', 'approval_month', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km']
_, datasets = load_datasets(project_root / 'data')
raw = datasets['train'].copy()
raw['period'] = pd.to_datetime(raw.RENT_APPROVAL_DATE + '-01').dt.to_period('M')
periods = np.sort(raw.period.unique())
cutoff = periods[int(len(periods) * 0.8)]
raw_fit = raw[raw.period < cutoff].copy()
raw_valid = raw[raw.period >= cutoff].copy()
print(f'Cutoff={cutoff}; fit={len(raw_fit):,}; validation={len(raw_valid):,}; epochs={EPOCHS}')


Cutoff=2024-05; fit=117,988; validation=32,012; epochs=80


In [2]:
class FoldEncoder:

    def fit(self, x, y):
        self.median = x[NUMS].median()
        self.mean = x[NUMS].fillna(self.median).mean()
        self.std = x[NUMS].fillna(self.median).std().replace(0, 1)
        self.vocabs = {c: {v: i + 1 for i, v in enumerate(x[c].astype(str).unique())} for c in CATS}
        self.ymean = float(y.mean())
        self.ystd = float(y.std()) or 1.0
        return self

    def transform(self, x):
        numeric = ((x[NUMS].fillna(self.median) - self.mean) / self.std).to_numpy(np.float32)
        cats = np.stack([x[c].astype(str).map(self.vocabs[c]).fillna(0).to_numpy(np.int64) for c in CATS], 1)
        return (numeric, cats)

class TabularNet(nn.Module):

    def __init__(self, cardinalities, width, dropout):
        super().__init__()
        dims = [min(32, max(4, (n + 1) // 2)) for n in cardinalities]
        self.emb = nn.ModuleList([nn.Embedding(n + 1, d) for n, d in zip(cardinalities, dims)])
        size = len(NUMS) + sum(dims)
        self.net = nn.Sequential(nn.Linear(size, width), nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, width // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width // 2, 1))

    def forward(self, n, c):
        return self.net(torch.cat([n] + [e(c[:, i]) for i, e in enumerate(self.emb)], 1)).squeeze(1)

class FTTransformerNet(nn.Module):
    """Feature-token Transformer for categorical and standardized numeric inputs."""

    def __init__(self, cardinalities, token_dim=32, layers=2):
        super().__init__()
        self.emb = nn.ModuleList([nn.Embedding(n + 1, token_dim) for n in cardinalities])
        self.numeric_weight = nn.Parameter(torch.empty(len(NUMS), token_dim))
        self.numeric_bias = nn.Parameter(torch.empty(len(NUMS), token_dim))
        self.cls_token = nn.Parameter(torch.empty(1, 1, token_dim))
        nn.init.normal_(self.numeric_weight, std=0.02)
        nn.init.normal_(self.numeric_bias, std=0.02)
        nn.init.normal_(self.cls_token, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=token_dim, nhead=4, dim_feedforward=token_dim * 4, dropout=0.15, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=layers)
        self.head = nn.Sequential(nn.LayerNorm(token_dim), nn.Linear(token_dim, 96), nn.GELU(), nn.Dropout(0.15), nn.Linear(96, 1))

    def forward(self, n, c):
        categorical_tokens = torch.stack([e(c[:, i]) for i, e in enumerate(self.emb)], dim=1)
        numeric_tokens = n.unsqueeze(-1) * self.numeric_weight.unsqueeze(0) + self.numeric_bias.unsqueeze(0)
        cls_token = self.cls_token.expand(n.shape[0], -1, -1)
        encoded = self.transformer(torch.cat([cls_token, categorical_tokens, numeric_tokens], dim=1))
        return self.head(encoded[:, 0]).squeeze(1)
EXPERTS = {'wide_mlp': {'kind': 'mlp', 'width': 192, 'dropout': 0.1}, 'regularized_mlp': {'kind': 'mlp', 'width': 96, 'dropout': 0.3}, 'ft_transformer': {'kind': 'ft_transformer'}}

def train_expert(train_n, train_c, y, stop_n, stop_c, stop_y, cardinalities, config):
    if config['kind'] == 'ft_transformer':
        model, learning_rate, weight_decay = (FTTransformerNet(cardinalities), 0.0003, 0.001)
    else:
        model = TabularNet(cardinalities, config['width'], config['dropout'])
        learning_rate, weight_decay = (0.001, 0.0001)
    opt = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(train_n, train_c, y), batch_size=BATCH, shuffle=True)
    best, bestloss, stale = (None, np.inf, 0)
    for epoch in range(EPOCHS):
        model.train()
        for n, c, t in loader:
            loss = ((model(n, c) - t) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            loss = ((model(stop_n, stop_c) - stop_y) ** 2).mean().item()
        if loss < bestloss - 1e-05:
            bestloss, best, stale = (loss, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0)
        else:
            stale += 1
            if stale >= 12:
                break
    model.load_state_dict(best)
    return model

def fold_matrices(early, later):
    p = RentalPreprocessor(datasets)
    a = p.fit_transform(early.drop(columns=['period', TARGET]))
    b = p.transform(later.drop(columns=['period', TARGET]))
    return (a[CATS + NUMS], b[CATS + NUMS])


In [3]:
fit_periods = np.sort(raw_fit.period.unique())
oof_parts = []
for fold, (tr, va) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods), 1):
    early = raw_fit[raw_fit.period.isin(fit_periods[tr])].copy()
    later = raw_fit[raw_fit.period.isin(fit_periods[va])].copy()
    xtr, xva = fold_matrices(early, later)
    ytr = early[TARGET].astype(float).reset_index(drop=True)
    yva = later[TARGET].astype(float).reset_index(drop=True)
    enc = FoldEncoder().fit(xtr, ytr)
    trn, trc = enc.transform(xtr)
    van, vac = enc.transform(xva)
    trn, trc, van, vac = [torch.tensor(v, dtype=torch.float32 if i in [0, 2] else torch.long) for i, v in enumerate([trn, trc, van, vac])]
    ytrz = torch.tensor(((ytr - enc.ymean) / enc.ystd).to_numpy(), dtype=torch.float32)
    yvaz = torch.tensor(((yva - enc.ymean) / enc.ystd).to_numpy(), dtype=torch.float32)
    out = xva[['BLOCK', 'TOWN']].reset_index(drop=True).copy()
    out['target'] = yva
    out['fold'] = fold
    cards = [len(enc.vocabs[c]) for c in CATS]
    for name, config in EXPERTS.items():
        model = train_expert(trn, trc, ytrz, van, vac, yvaz, cards, config)
        model.eval()
        with torch.no_grad():
            out[name] = model(van, vac).numpy() * enc.ystd + enc.ymean
    oof_parts.append(out)
    print(f'OOF fold {fold}: {len(out):,} rows')
oof = pd.concat(oof_parts, ignore_index=True)
assert oof[list(EXPERTS)].notna().all().all()


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


/var/folders/xd/2yk04_yn3ls3kqgyz397mjyh0000gn/T/ipykernel_12880/2516232820.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=layers)


OOF fold 1: 28,514 rows
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash

/var/folders/xd/2yk04_yn3ls3kqgyz397mjyh0000gn/T/ipykernel_12880/2516232820.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=layers)


OOF fold 2: 27,858 rows
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash

/var/folders/xd/2yk04_yn3ls3kqgyz397mjyh0000gn/T/ipykernel_12880/2516232820.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=layers)


OOF fold 3: 30,051 rows


In [4]:
stop_fold = oof.fold.max()
gate_train = oof[oof.fold < stop_fold].copy()
gate_stop = oof[oof.fold == stop_fold].copy()

def vocab(x):
    return {v: i + 1 for i, v in enumerate(sorted(x.dropna().astype(str).unique()))}
bv, tv = (vocab(gate_train.BLOCK), vocab(gate_train.TOWN))

def idx(x, col, v):
    return torch.tensor(x[col].astype(str).map(v).fillna(0).astype(int).to_numpy(), dtype=torch.long)
pred_mean = gate_train[list(EXPERTS)].mean()
pred_std = gate_train[list(EXPERTS)].std().replace(0, 1)

def gate_preds(x):
    return torch.tensor(((x[list(EXPERTS)] - pred_mean) / pred_std).to_numpy(), dtype=torch.float32)

class Gate(nn.Module):

    def __init__(self):
        super().__init__()
        self.b = nn.Embedding(len(bv) + 1, 16)
        self.t = nn.Embedding(len(tv) + 1, 8)
        self.net = nn.Sequential(nn.Linear(16 + 8 + len(EXPERTS), 48), nn.ReLU(), nn.Dropout(0.1), nn.Linear(48, len(EXPERTS)))

    def forward(self, b, t, scaled, raw):
        w = torch.softmax(self.net(torch.cat([self.b(b), self.t(t), scaled], 1)), 1)
        return ((w * raw).sum(1), w)
gate = Gate()
opt = torch.optim.AdamW(gate.parameters(), lr=0.001)
loader = DataLoader(TensorDataset(idx(gate_train, 'BLOCK', bv), idx(gate_train, 'TOWN', tv), gate_preds(gate_train), torch.tensor(gate_train[list(EXPERTS)].to_numpy(), dtype=torch.float32), torch.tensor(gate_train.target.to_numpy(), dtype=torch.float32)), batch_size=BATCH, shuffle=True)
best, bestloss, stale = (None, np.inf, 0)
for epoch in range(200):
    gate.train()
    for b, t, s, r, y in loader:
        p, _ = gate(b, t, s, r)
        loss = ((p - y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    gate.eval()
    with torch.no_grad():
        p, _ = gate(idx(gate_stop, 'BLOCK', bv), idx(gate_stop, 'TOWN', tv), gate_preds(gate_stop), torch.tensor(gate_stop[list(EXPERTS)].to_numpy(), dtype=torch.float32))
        loss = ((p - torch.tensor(gate_stop.target.to_numpy(), dtype=torch.float32)) ** 2).mean().item()
    if loss < bestloss - 0.001:
        bestloss, best, stale = (loss, {k: v.detach().clone() for k, v in gate.state_dict().items()}, 0)
    else:
        stale += 1
        if stale >= 20:
            break
gate.load_state_dict(best)
xtr, xva = fold_matrices(raw_fit, raw_valid)
ytr = raw_fit[TARGET].astype(float).reset_index(drop=True)
enc = FoldEncoder().fit(xtr, ytr)
trn, trc = enc.transform(xtr)
van, vac = enc.transform(xva)
trn, trc, van, vac = [torch.tensor(v, dtype=torch.float32 if i in [0, 2] else torch.long) for i, v in enumerate([trn, trc, van, vac])]
ytrz = torch.tensor(((ytr - enc.ymean) / enc.ystd).to_numpy(), dtype=torch.float32)
hold = xva[['BLOCK', 'TOWN']].reset_index(drop=True).copy()
cards = [len(enc.vocabs[c]) for c in CATS]
for name, config in EXPERTS.items():
    model = train_expert(trn, trc, ytrz, trn, trc, ytrz, cards, config)
    model.eval()
    with torch.no_grad():
        hold[name] = model(van, vac).numpy() * enc.ystd + enc.ymean
with torch.no_grad():
    final_pred, weights = gate(idx(hold, 'BLOCK', bv), idx(hold, 'TOWN', tv), gate_preds(hold), torch.tensor(hold[list(EXPERTS)].to_numpy(), dtype=torch.float32))
actual = raw_valid[TARGET].astype(float).reset_index(drop=True)
hold['gated'] = final_pred.numpy()
[hold.__setitem__(f'{name}_weight', weights[:, i].numpy()) for i, name in enumerate(EXPERTS)]
comparison = pd.DataFrame([{'model': name, 'RMSE': mean_squared_error(actual, hold[name]) ** 0.5, 'MAE': mean_absolute_error(actual, hold[name])} for name in EXPERTS] + [{'model': 'equal blend', 'RMSE': mean_squared_error(actual, hold[list(EXPERTS)].mean(1)) ** 0.5, 'MAE': mean_absolute_error(actual, hold[list(EXPERTS)].mean(1))}, {'model': 'block→town DL gate', 'RMSE': mean_squared_error(actual, hold.gated) ** 0.5, 'MAE': mean_absolute_error(actual, hold.gated)}])
display(comparison.sort_values('RMSE'))
display(hold[[f'{n}_weight' for n in EXPERTS]].describe().T)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


/var/folders/xd/2yk04_yn3ls3kqgyz397mjyh0000gn/T/ipykernel_12880/2516232820.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=layers)
/var/folders/xd/2yk04_yn3ls3kqgyz397mjyh0000gn/T/ipykernel_12880/835276702.py:69: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating th

,model,RMSE,MAE
3,equal blend,471.168036,340.981239
4,block→town DL gate,475.129793,343.592058
2,ft_transformer,476.979104,347.919657
1,regularized_mlp,482.871008,350.043880
0,wide_mlp,496.308418,365.135888


,count,mean,std,min,25%,50%,75%,max
wide_mlp_weight,32012.0,0.350870,0.081993,0.118418,0.293351,0.351327,0.407353,0.687491
regularized_mlp_weight,32012.0,0.526122,0.084625,0.256397,0.464379,0.519872,0.582931,0.842627
ft_transformer_weight,32012.0,0.123009,0.032542,0.028328,0.099607,0.121481,0.144959,0.250241
